**1️⃣Create Widget**

In [0]:
dbutils.widgets.text("source_system", "")
source_system = dbutils.widgets.get("source_system")

if not source_system:
    raise ValueError("source_system cannot be empty")

print(f"Source system: {source_system}")

Source system: delta


2️⃣ Read Metadata Tables

In [0]:
tables_df = spark.table("banking.metadata.tables")

filtered_df = (
    tables_df
    .filter(f"active_flag = true AND lower(source_system) = lower('{source_system}')")
    .orderBy("load_order")
)

display(filtered_df)

table_id,table_name,source_system,source_schema,source_table,source_path,target_layer,bronze_schema,silver_schema,gold_schema,active_flag,load_order,created_at
1,customers,delta,banking,customers,banking.banking.customers,silver,bronze,silver,null,true,1,2026-05-12T12:27:25.121Z
2,accounts,delta,banking,accounts,banking.banking.accounts,silver,bronze,silver,null,true,2,2026-05-12T12:27:25.121Z
3,transactions,delta,banking,transactions,banking.banking.transactions,silver,bronze,silver,null,true,3,2026-05-12T12:27:25.121Z
4,branches,delta,banking,branches,banking.banking.branches,silver,bronze,silver,null,true,4,2026-05-12T12:27:25.121Z


3️⃣ Convert to List of Dictionaries

In [0]:
rows = filtered_df.collect()
tables_list = [row.asDict() for row in rows]

print(f"Total tables to process: {len(tables_list)}")
print(tables_list)

Total tables to process: 4
[{'table_id': 1, 'table_name': 'customers', 'source_system': 'delta', 'source_schema': 'banking', 'source_table': 'customers', 'source_path': 'banking.banking.customers', 'target_layer': 'silver', 'bronze_schema': 'bronze', 'silver_schema': 'silver', 'gold_schema': None, 'active_flag': True, 'load_order': 1, 'created_at': datetime.datetime(2026, 5, 12, 12, 27, 25, 121306)}, {'table_id': 2, 'table_name': 'accounts', 'source_system': 'delta', 'source_schema': 'banking', 'source_table': 'accounts', 'source_path': 'banking.banking.accounts', 'target_layer': 'silver', 'bronze_schema': 'bronze', 'silver_schema': 'silver', 'gold_schema': None, 'active_flag': True, 'load_order': 2, 'created_at': datetime.datetime(2026, 5, 12, 12, 27, 25, 121306)}, {'table_id': 3, 'table_name': 'transactions', 'source_system': 'delta', 'source_schema': 'banking', 'source_table': 'transactions', 'source_path': 'banking.banking.transactions', 'target_layer': 'silver', 'bronze_schema': '

In [0]:
from pyspark.sql.functions import col, cast

In [0]:
rows = filtered_df.collect()
tables_list = []

for row in rows:
    row_dict = row.asDict()
    row_dict["created_at"] = str(row_dict["created_at"])
    tables_list.append(row_dict)

print(f"Total tables to process: {len(tables_list)}")
print(tables_list)

Total tables to process: 4
[{'table_id': 1, 'table_name': 'customers', 'source_system': 'delta', 'source_schema': 'banking', 'source_table': 'customers', 'source_path': 'banking.banking.customers', 'target_layer': 'silver', 'bronze_schema': 'bronze', 'silver_schema': 'silver', 'gold_schema': None, 'active_flag': True, 'load_order': 1, 'created_at': '2026-05-12 12:27:25.121306'}, {'table_id': 2, 'table_name': 'accounts', 'source_system': 'delta', 'source_schema': 'banking', 'source_table': 'accounts', 'source_path': 'banking.banking.accounts', 'target_layer': 'silver', 'bronze_schema': 'bronze', 'silver_schema': 'silver', 'gold_schema': None, 'active_flag': True, 'load_order': 2, 'created_at': '2026-05-12 12:27:25.121306'}, {'table_id': 3, 'table_name': 'transactions', 'source_system': 'delta', 'source_schema': 'banking', 'source_table': 'transactions', 'source_path': 'banking.banking.transactions', 'target_layer': 'silver', 'bronze_schema': 'bronze', 'silver_schema': 'silver', 'gold_sc

In [0]:
dbutils.jobs.taskValues.set(
    key="tables_metadata",
    value=tables_list
)

print("Task value set successfully.")

Task value set successfully.


In [0]:
# # %sql
# -- SELECT t.table_id, t.table_name, p.parameter_name, p.parameter_value
# -- FROM banking.metadata.tables t
# -- JOIN banking.metadata.table_parameters p ON t.table_id = p.table_id
# -- WHERE t.table_id IN (1,2,3,4)
# -- ORDER BY t.table_id, p.parameter_name

table_id,table_name,parameter_name,parameter_value
1,customers,load_type,MERGE
1,customers,primary_key,customer_id
1,customers,watermark_column,updated_at
2,accounts,load_type,MERGE
2,accounts,primary_key,account_id
2,accounts,watermark_column,updated_at
3,transactions,load_type,APPEND
3,transactions,primary_key,txn_id
3,transactions,watermark_column,txn_timestamp
4,branches,load_type,FULL


In [0]:
# %sql
# SELECT * FROM banking.metadata.table_watermarks
# WHERE table_id IN (1,2,3,4)

table_id,last_watermark_value,last_updated_at,last_run_id
1,1900-01-01 00:00:00,2026-05-12T12:27:53.111Z,null
2,1900-01-01 00:00:00,2026-05-12T12:27:53.111Z,null
3,1900-01-01 00:00:00,2026-05-12T12:27:53.111Z,null


In [0]:
%sql
SELECT * FROM banking.metadata.table_watermarks WHERE table_id = 1

table_id,last_watermark_value,last_updated_at,last_run_id
1,1900-01-01 00:00:00,2026-05-12T12:27:53.111Z,null
